# Ordered Logistic Regression Results for Adoption Predictors (FAIR²) Exploration with `mlcroissant`
This notebook provides an example workflow for loading, exploring, and processing the 
[FAIR² dataset: Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya](https://sen.science/doi/10.71728/senscience.y7m0-f273).

We use the `mlcroissant` library to access data defined by its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Citation: {metadata.citeAs}")

## 2. Data Overview
Review available record sets (`@id`s), their fields, and key properties, referencing all by `@id`.

In [ ]:
# List the dataset's record sets by `@id`
record_sets = list(dataset.record_sets())
if len(record_sets) == 0:
    print("No record sets declared at the top-level 'recordSet' field. Scanning for available record sets...")
    # Alternative approach: Explore all available record sets via the records API
    # Using private API to enumerate record sets, as mlcroissant >=0.7 provides .record_sets()
    rs_ids = [rs['@id'] for rs in dataset._metadata_json.get('recordSet', [])]
    if not rs_ids:
        # Fallback: try another way (if none registered, show only distribution files)
        print("No record sets declared in schema. The dataset may be only metadata or the schema may embed record sets elsewhere.")
        record_sets = []
    else:
        record_sets = rs_ids

if not record_sets:
    print("No record sets discovered in this Croissant package. If this is unexpected, check the schema definition and consider querying underlying distribution(s) directly.")
else:
    print("Discovered record sets:")
    for rid in record_sets:
        print(f"  - {rid}")

# For demonstration, let's try to enumerate record sets via dataset.record_sets() (works if present)
for rs in getattr(dataset, "record_sets", lambda: [])():
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            fname = field.get('name', '')
            print(f"    - @id: {field['@id']} (name: {fname})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If record sets are missing or not resolved in the schema, extraction may not succeed – see comments below.

In [ ]:
# Attempt to extract data from available record sets
import warnings

dfs = {}

# Check for record sets
if not record_sets:
    print("No record sets to extract.\nIf a 'recordSet' is missing from the top-level metadata, data may be accessible only via 'distribution' or not publicly available.")
else:
    print("Extracting tabular records for each record set by @id...")
    for rs_id in record_sets:
        try:
            records_iter = dataset.records(record_set=rs_id)
            records = list(records_iter)
            if not records:
                print(f"  No records found in record set {rs_id}")
            else:
                df = pd.DataFrame(records)
                dfs[rs_id] = df
                print(f"  Loaded {len(df)} records from record set {rs_id}.")
        except Exception as e:
            warnings.warn(f"Could not load records for {rs_id}: {e}")

# If any DataFrames were loaded, display their columns and preview
if dfs:
    for k, v in dfs.items():
        print(f"\nColumns for record set {k}:")
        print(v.columns.tolist())
        display(v.head())
        break  # Show only one as example
else:
    print("No tabular data could be loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, and grouping.

**Note:** Replace the `example_recordset_id` and field IDs below with the appropriate `@id`s from your dataset if available.

In [ ]:
# EXAMPLE: Replace 'example_recordset_id' and field IDs with actual @id values from your dataset

# Find the first available DataFrame
if dfs:
    # Pick the first record set
    record_set_id = list(dfs.keys())[0]
    df = dfs[record_set_id]
    
    # Attempt to find a numeric field
    numeric_field = None
    for col in df.columns:
        # Heuristic: look for typical numeric column names
        if any(x in col.lower() for x in ['value', 'score', 'coefficient', 'error', 'p', 'log', 'iteration']) and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        print("Could not automatically identify a numeric field. Please refer to the record set documentation and provide a numeric field @id.")
    else:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Attempt grouping by a likely categorical field
        group_field = None
        for col in df.columns:
            if any(x in col.lower() for x in ['category', 'type', 'ward', 'region', 'county', 'gender', 'group']) and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical group field identified.")
else:
    print("No DataFrames loaded; cannot proceed with EDA.")

## 5. Visualization
Visualize distributions or relationships between available numeric and categorical fields.

_Replace example field names with record set field `@id`s as appropriate._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dfs and numeric_field:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping is possible
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df, showfliers=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion
In this notebook, we:
- Loaded dataset metadata and, when available, tabular record sets using the Croissant schema and `mlcroissant`
- Provided an overview of available record sets and their fields, referencing all by `@id`
- Demonstrated data extraction into `pandas` DataFrames and showcased filtering, normalization, and grouping steps
- Illustrated exploratory visualization of numeric data fields using Seaborn/Matplotlib

This workflow can be adapted for other FAIR² or Croissant-standard datasets. For further analysis, refer to the dataset documentation or schema for field-specific details and domain expertise.
